Baseline model:
- Filter train/val departures by [1, 99]-th percentiles (around 5 and 40 minutes)
- Use last available month (Dec. 2025) as a validation set
- No features engineering, arrival movements are not used
- Used features:
    - FLIGHT_RULE_mvt
    - ADEP_mvt
    - AIRCRAFT_TYPE_mvt
    - RUNWAY_mvt
    - STAND_mvt
    - MARKET_SEGMENT_flt
    - FLIGHT_RULE_flt
    - FLIGHT_TYPE_flt
    - AIRCRAFT_TYPE_flt
    - WK_TBL_CAT_flt
- Target: TAXITIME_SEC_mvt
- Model: CatBoost

In [1]:
import os

import optuna
import pandas as pd
from catboost import CatBoostRegressor

/opt/miniconda3/envs/ml/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Read training data

In [2]:
def read_train(data_dir: str) -> pd.DataFrame:
    filenames = [x for x in os.listdir(data_dir) if x.startswith("training")]
    dfs = []
    for filename in filenames:
        month = int(filename.split("_")[1].split("-")[1])
        df = pd.read_parquet(os.path.join(data_dir, filename))
        df["month"] = month
        dfs.append(df)
    return pd.concat(dfs, ignore_index=True)

In [3]:
df = read_train(data_dir="../data")
df.shape

(4167797, 31)

### Drop arrivals

In [4]:
df = df[df["PHASE_mvt"] == "DEP"]
df.shape

(2085047, 31)

### Features/target

In [5]:
cat_features = [
    "FLIGHT_RULE_mvt",
    "ADEP_mvt",
    "AIRCRAFT_TYPE_mvt",
    "RUNWAY_mvt",
    "STAND_mvt",
    "MARKET_SEGMENT_flt",
    "FLIGHT_RULE_flt",
    "FLIGHT_TYPE_flt",
    "AIRCRAFT_TYPE_flt",
    "WK_TBL_CAT_flt"
]
target = "TAXITIME_SEC_mvt"

### Fill NaN values in categorical features

In [6]:
df[cat_features] = df[cat_features].fillna("_NAN_")
df[cat_features].isna().sum()

FLIGHT_RULE_mvt       0
ADEP_mvt              0
AIRCRAFT_TYPE_mvt     0
RUNWAY_mvt            0
STAND_mvt             0
MARKET_SEGMENT_flt    0
FLIGHT_RULE_flt       0
FLIGHT_TYPE_flt       0
AIRCRAFT_TYPE_flt     0
WK_TBL_CAT_flt        0
dtype: int64

### Split to train/val

In [7]:
df_train = df[df["month"] != 12]
df_val = df[df["month"] == 12]
df_train.shape, df_val.shape

((1919370, 31), (165677, 31))

### Drop train/val departures by [1, 99] range

In [8]:
q_1 = df_train["TAXITIME_SEC_mvt"].quantile(0.01)
q_99 = df_train["TAXITIME_SEC_mvt"].quantile(0.99)
df_train = df_train[(df_train["TAXITIME_SEC_mvt"] >= q_1) & (df_train["TAXITIME_SEC_mvt"] <= q_99)]
df_val = df_val[(df_val["TAXITIME_SEC_mvt"] >= q_1) & (df_val["TAXITIME_SEC_mvt"] <= q_99)]
print(f"1st percentile: {q_1}, 99th percentile: {q_99}")
df_train.shape, df_val.shape

1st percentile: 291.0, 99th percentile: 2339.0


((1881066, 31), (162569, 31))

### Hyperparameters tuning

In [9]:
def objective(trial):
    params = {
        "iterations": trial.suggest_int("iterations", 10, 100),
        "depth": trial.suggest_int("depth", 4, 10),
        "learning_rate": trial.suggest_float("learning_rate", 1e-3, 1e-1, log=True),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1e-3, 10.0, log=True),
        "random_strength": trial.suggest_float("random_strength", 1e-3, 10.0, log=True),
        "bagging_temperature": trial.suggest_float("bagging_temperature", 0.0, 1.0),
        "border_count": trial.suggest_int("border_count", 32, 255),
        "verbose": False,
    }

    model = CatBoostRegressor(**params, cat_features=cat_features)
    model.fit(
        X=df_train[cat_features],
        y=df_train[target],
        eval_set=(df_val[cat_features], df_val[target]),
        early_stopping_rounds=50,
    )
    train_preds = model.predict(df_train[cat_features])
    train_rmse = ((train_preds - df_train[target]) ** 2).mean() ** 0.5
    val_preds = model.predict(df_val[cat_features])
    val_rmse = ((val_preds - df_val[target]) ** 2).mean() ** 0.5
    print(f"Trial {trial.number}: train RMSE = {train_rmse}, val RMSE = {val_rmse}")

    return val_rmse


In [10]:
study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=50)
study.best_params

[I 2026-10-05 03:51:33,845] A new study created in memory with name: no-name-1776af34-08fe-4559-b02a-d101638f7d40
[I 2026-10-05 03:51:52,023] Trial 0 finished with value: 333.07912672479546 and parameters: {'iterations': 76, 'depth': 8, 'learning_rate': 0.0029874748132801563, 'l2_leaf_reg': 0.06028500866438588, 'random_strength': 4.717683545601787, 'bagging_temperature': 0.3810275650099261, 'border_count': 137}. Best is trial 0 with value: 333.07912672479546.


Trial 0: train RMSE = 327.33037337587183, val RMSE = 333.07912672479546


[I 2026-10-05 03:52:10,417] Trial 1 finished with value: 288.3724580415548 and parameters: {'iterations': 85, 'depth': 7, 'learning_rate': 0.020680168736501605, 'l2_leaf_reg': 0.020369542783136314, 'random_strength': 5.640651253144987, 'bagging_temperature': 0.07976480118930296, 'border_count': 52}. Best is trial 1 with value: 288.3724580415548.


Trial 1: train RMSE = 275.9330439643602, val RMSE = 288.3724580415548


[I 2026-10-05 03:52:30,781] Trial 2 finished with value: 275.13359822351106 and parameters: {'iterations': 82, 'depth': 8, 'learning_rate': 0.08383646726509142, 'l2_leaf_reg': 0.014577699028418227, 'random_strength': 0.001187697479917588, 'bagging_temperature': 0.007204347721957216, 'border_count': 150}. Best is trial 2 with value: 275.13359822351106.


...

[I 2026-10-05 04:20:13,397] Trial 48 finished with value: 272.04821773432207 and parameters: {'iterations': 99, 'depth': 10, 'learning_rate': 0.08641249769425964, 'l2_leaf_reg': 0.0015139690679216713, 'random_strength': 0.014247310964315509, 'bagging_temperature': 0.12169344153288009, 'border_count': 216}. Best is trial 23 with value: 271.6540726017829.


Trial 48: train RMSE = 256.1412155516121, val RMSE = 272.04821773432207


[I 2026-10-05 04:20:55,444] Trial 49 finished with value: 273.029591110972 and parameters: {'iterations': 88, 'depth': 10, 'learning_rate': 0.07485770184774469, 'l2_leaf_reg': 0.006218936133839934, 'random_strength': 0.0026278975840779675, 'bagging_temperature': 0.14490676580947834, 'border_count': 161}. Best is trial 23 with value: 271.6540726017829.


Trial 49: train RMSE = 257.38032638245807, val RMSE = 273.029591110972


{'iterations': 90,
 'depth': 10,
 'learning_rate': 0.09706147073046165,
 'l2_leaf_reg': 0.001589421901393926,
 'random_strength': 0.005455570470491138,
 'bagging_temperature': 0.29605742867371637,
 'border_count': 235}

### Refit on full data

In [14]:
df = df[(df["TAXITIME_SEC_mvt"] >= q_1) & (df["TAXITIME_SEC_mvt"] <= q_99)]
submit_model = CatBoostRegressor(**study.best_params, cat_features=cat_features)
submit_model.fit(
    X=df[cat_features],
    y=df[target],
    early_stopping_rounds=50,
)

0:	learn: 337.5317708	total: 346ms	remaining: 30.8s
1:	learn: 328.5672827	total: 614ms	remaining: 27s
2:	learn: 321.0739608	total: 851ms	remaining: 24.7s
3:	learn: 314.7041165	total: 1.07s	remaining: 23.1s
4:	learn: 309.2806866	total: 1.31s	remaining: 22.3s
5:	learn: 304.8208565	total: 1.56s	remaining: 21.8s
6:	learn: 301.1346255	total: 1.78s	remaining: 21.1s
7:	learn: 297.5210767	total: 2s	remaining: 20.5s
8:	learn: 292.6802057	total: 2.23s	remaining: 20.1s
9:	learn: 288.4648180	total: 2.44s	remaining: 19.5s
10:	learn: 285.0093041	total: 2.67s	remaining: 19.2s
11:	learn: 282.0371996	total: 2.88s	remaining: 18.7s
12:	learn: 279.5310907	total: 3.14s	remaining: 18.6s
13:	learn: 277.4394342	total: 3.37s	remaining: 18.3s
14:	learn: 275.6859807	total: 3.59s	remaining: 17.9s
15:	learn: 274.2371547	total: 3.83s	remaining: 17.7s
16:	learn: 272.9022098	total: 4.08s	remaining: 17.5s
17:	learn: 271.8095107	total: 4.31s	remaining: 17.2s
18:	learn: 270.8472116	total: 4.53s	remaining: 16.9s
19:	lear

CatBoostRegressor(bagging_temperature=0.29605742867371637, border_count=235, cat_features=['FLIGHT_RULE_mvt', 'ADEP_mvt', 'AIRCRAFT_TYPE_mvt', 'RUNWAY_mvt', 'STAND_mvt', 'MARKET_SEGMENT_flt', 'FLIGHT_RULE_flt', 'FLIGHT_TYPE_flt', 'AIRCRAFT_TYPE_flt', 'WK_TBL_CAT_flt'], depth=10, iterations=90, l2_leaf_reg=0.001589421901393926, learning_rate=0.09706147073046165, loss_function='RMSE', random_strength=0.005455570470491138)

### Features importances

In [25]:
importances = dict(zip(cat_features, submit_model.feature_importances_))
dict(sorted(importances.items(), key=lambda item: item[1]))

{'FLIGHT_RULE_mvt': np.float64(0.0),
 'FLIGHT_RULE_flt': np.float64(0.0),
 'FLIGHT_TYPE_flt': np.float64(1.0016181676754876),
 'AIRCRAFT_TYPE_mvt': np.float64(1.4780021376578067),
 'AIRCRAFT_TYPE_flt': np.float64(5.746726360328671),
 'MARKET_SEGMENT_flt': np.float64(7.5912624064423575),
 'WK_TBL_CAT_flt': np.float64(10.946562576803966),
 'STAND_mvt': np.float64(21.040198382637378),
 'ADEP_mvt': np.float64(25.654573121088845),
 'RUNWAY_mvt': np.float64(26.54105684736548)}

### Make predictions for submit

In [15]:
df_test = pd.read_parquet("../data/ranking.parquet")
df_test = df_test[df_test["PHASE_mvt"] == "DEP"]
df_test.shape

(344841, 30)

In [17]:
df_test[cat_features] = df_test[cat_features].fillna("_NAN_")
df_test[cat_features].isna().sum()

FLIGHT_RULE_mvt       0
ADEP_mvt              0
AIRCRAFT_TYPE_mvt     0
RUNWAY_mvt            0
STAND_mvt             0
MARKET_SEGMENT_flt    0
FLIGHT_RULE_flt       0
FLIGHT_TYPE_flt       0
AIRCRAFT_TYPE_flt     0
WK_TBL_CAT_flt        0
dtype: int64

In [18]:
test_preds = submit_model.predict(df_test[cat_features])
test_preds[:5]

array([1107.3991599 ,  938.67136167,  957.15608631,  908.03393664,
        675.35252263])

In [20]:
submit = pd.DataFrame({
    "MVT_ID_mvt": df_test["MVT_ID_mvt"],
    "TAXITIME_SEC_mvt": test_preds
})
submit.to_parquet("incredible-rose_v1.parquet")
submit.shape

(344841, 2)